<a href="https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# 1) My lane as an ML task

## Lane: Refresh / Content Opportunity Scoring

### ML task type: Ranking

I frame this problem as a ranking task. The goal is not simply to predict whether a page will decline. Instead, the goal is to rank content pages by their priority for human review and possible refresh.

The output would be a priority score for each page. Pages with higher scores would appear earlier in the review queue.

This is useful because the content team has limited time and cannot manually review every page at once.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# 2) Target or proxy

The starter dataset provides an observed `trend_direction` field. I will use this as a proxy for identifying pages that may deserve review.

For an initial framing, a page with a downward trend can be treated as a potential refresh opportunity. However, this is only a proxy for the real business outcome.

The true business outcome would be whether a content refresh actually improves the page's future performance. The starter data does not establish that causal relationship.

Therefore, I will avoid claiming that the target represents guaranteed future recovery.

### Proposed target

Target/proxy: `trend_direction`

Initial proxy definition:

- `down` → page shows a declining observed trend
- other trend values → page does not currently show the same decline signal


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# 3) Success metric

Because this is a ranking problem, the main success criterion should measure whether the highest-ranked pages contain a useful concentration of pages matching the review proxy.

A useful primary metric is **Precision@K**.

For example, Precision@100 would answer:

"Of the 100 pages placed at the top of the review queue, how many match the selected target/proxy?"

This metric matches the real action because the content team has limited review capacity. If the team can only review the top K pages, the quality of those top-ranked pages matters more than the performance of the entire dataset.

I would also compare the ranking against a simple fixed-rule baseline to determine whether ML actually improves prioritization.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
!git clone https://github.com/ubaid8878/Flyrank-ML-Internship.git


Cloning into 'Flyrank-ML-Internship'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 37), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.83 MiB | 12.47 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [3]:
import os

print(os.listdir("/content"))

['.config', 'Flyrank-ML-Internship', 'sample_data']


In [4]:
import pandas as pd

DATA_PATH = "/content/Flyrank-ML-Internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)

df.head()

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
# Show the columns available for the page-level analysis

print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
# Show a small page-level slice

page_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction"
]

page_df = df[page_cols].copy()

page_df.head(10)

,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down
6,content_9a34b442b552,client_8722616204,20,1,90,down
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down


In [ ]:
### Unit of analysis

The unit of analysis is **one content page**.

Therefore, each row in the dataframe represents one page and its observed performance signals.

For example, a row contains the page's content ID, client ID, search impressions, sessions, content age, and observed trend direction.

This unit of analysis matches the decision because the content team needs to decide which individual pages should be reviewed first.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [7]:
# Create a simple binary proxy target from trend_direction

page_df["target_proxy"] = (
    page_df["trend_direction"] == "down"
).astype(int)

page_df[[
    "content_id",
    "trend_direction",
    "target_proxy"
]].head(10)


,content_id,trend_direction,target_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


In [8]:
print(page_df["target_proxy"].value_counts())
print("\nProportion of pages with target proxy = 1:",
      round(page_df["target_proxy"].mean(), 3))

target_proxy
1    16262
0    13738
Name: count, dtype: int64

Proportion of pages with target proxy = 1: 0.542


In [10]:
### Target sketch

The `target_proxy` column converts the observed trend into a simple binary indicator:

- `1` means the page has an observed downward trend.
- `0` means it does not have an observed downward trend.

This is only a first target sketch. It should not be interpreted as proof that the page will recover if refreshed.

The target is useful for testing whether available page-level signals can help prioritize pages for human review.

SyntaxError: invalid syntax (4222546632.py, line 3)

In [ ]:
# 5) Why ML beats a fixed rule here

A fixed rule could be useful as a baseline. For example, we could simply rank pages by declining trend, low CTR, high impressions, or another single signal.

However, a fixed rule may miss interactions between several signals. A page with high impressions, declining trend, older content, and weak engagement may represent a different opportunity from a page that has only one of these characteristics.

ML could potentially learn combinations of multiple signals and produce a more useful ranking than a single manually chosen rule.

However, ML does not automatically beat a fixed rule. The correct approach is to build a simple baseline first and then test whether an ML-based ranking provides better Precision@K or another decision-relevant metric.

If ML does not outperform the baseline, the simpler rule may be preferable because it is easier to understand and maintain.

In [ ]:
## Action supported by the output

The ranking would be used by an SEO or content team as a review queue.

The team could start with the highest-ranked pages, inspect their content and search performance, and decide whether each page should be:

1. Refreshed,
2. Improved,
3. Left unchanged, or
4. Investigated further.

The model would therefore support prioritization rather than automatically deciding that a page must be refreshed.



```
`# This is formatted as code`
```

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.